In [1]:
import os
import hail as hl

In [27]:
from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc

In [3]:
# startup hail
hail_init.hail_bmrc_init('../logs/hail_grm.log', 'GRCh37')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe071.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to ../logs/hail_grm.log


In [4]:
# get markers used in UKBB kinship calculation
ht = hl.import_table('/well/lindgren/UKBIOBANK/DATA/QC/ukb_snp_qc.txt', impute = True, delimiter = ' ')
ht = ht.filter(ht.in_Relatedness == 1)
rsids = ht.rs_id.collect()

2021-09-09 11:36:45 Hail: WARN: Found 3 duplicate columns. Mangled columns follows:
  'PC9_loading' -> 'PC9_loading_1'
  'PC9_loading' -> 'PC9_loading_2'
  'PC9_loading' -> 'PC9_loading_3'
2021-09-09 11:36:45 Hail: INFO: Reading table to impute column types
2021-09-09 11:37:06 Hail: WARN: Found 3 duplicate columns. Mangled columns follows:
  'PC9_loading' -> 'PC9_loading_1'
  'PC9_loading' -> 'PC9_loading_2'
  'PC9_loading' -> 'PC9_loading_3'
2021-09-09 11:37:06 Hail: INFO: Loading 159 fields. Counts by type:
  int32: 113
  float64: 40
  str: 6


In [30]:
# combine multiple matrix tables
mt = genotypes.get_ukb_imputed_v3_bgen(['22'])
mt = mt.filter_rows(hl.literal(rsids).contains(mt.rsid))

2021-09-09 12:09:17 Hail: INFO: Number of BGEN files parsed: 1
2021-09-09 12:09:17 Hail: INFO: Number of samples in BGEN files: 487409
2021-09-09 12:09:17 Hail: INFO: Number of variants across all BGEN files: 1255683


In [31]:
# get white british ancestry
mt = qc.filter_to_european(mt)

2021-09-09 12:09:19 Hail: INFO: Reading table without type imputation
  Loading field 'eid' as type str (user-supplied)
  Loading field 'genetically_european' as type int32 (user-supplied)


[get_genetically_european]: Not all samples IDs mapped perfectly (125/487409 IDs are undefined)


KeyboardInterrupt: 

In [15]:
import os
import hail as hl
import argparse

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc

def main(args):

    hail_init.hail_bmrc_init('logs/hail_grm.log', 'GRCh37')
    
    # input variables
    chroms = args.chroms
    subset_markers_by_kinship = args.subset_markers_by_kinship
    out_prefix = args.out_prefix
    subset_samples_by_eur = args.subset_samples_by_eur
        
    # combine multiple matrix tables
    mt = genotypes.get_ukb_imputed_v3_bgen(chroms)
       
    # createse a sparse GRM
    if subset_markers_by_kinship:
        ht = hl.import_table('/well/lindgren/UKBIOBANK/DATA/QC/ukb_snp_qc.txt', impute = True, delimiter = ' ')
        ht = ht.filter(ht.in_Relatedness == 1)
        rsids = ht.rs_id.collect()
        mt = mt.filter_rows(hl.literal(rsids).contains(mt.rsid))
    
    # get white british ancestry
    if subset_samples_by_eur:
        mt = qc.filter_to_european(mt)
    
    hl.export_plink(mt, out_prefix, ind_id = mt.s)
    
if __name__=='__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--chroms', nargs='+', default=None, help='chroms to be included')
    parser.add_argument('--subset_markers_by_kinship', action='store_true', help='Only use markers that have been used in UKBB as kinship markers')
    parser.add_argument('--subset_samples_by_eur', action='store_true', help='Subset samples to those that are genetically european.')
    parser.add_argument('--out_prefix', default=None, help='Path prefix for output dataset (plink format)')
    args = parser.parse_args()

    main(args)
    

usage: ipykernel_launcher.py [-h] [--chroms CHROMS [CHROMS ...]]
                             [--subset_markers_by_kinship]
                             [--subset_samples_by_eur]
                             [--out_prefix OUT_PREFIX]
ipykernel_launcher.py: error: unrecognized arguments: -f /gpfs3/users/gms/mmq446/.local/share/jupyter/runtime/kernel-1a459069-fcf1-432a-ab35-256208b919be.json


SystemExit: 2

/well/lindgren/users/mmq446/conda/skylake/envs/jupyter-hail/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3351: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
